## **MODELADO.SQL — Esquema Estrella**

Se construye un modelo dimensional en esquema estrella compuesto por:

| Tabla | Tipo | Descripción |
|---|---|---|
| `dim_categoria` | Dimensión | Categorías únicas de producto |
| `dim_fecha` | Dimensión | Calendario con año, mes y día |
| `dim_genero` | Dimensión | Géneros únicos de clientes |
| `Hechos_Ventas` | Hechos | Transacciones con métricas de venta |

**Decisión de diseño:** las dimensiones se generan con `ROW_NUMBER()` para crear 
claves surrogadas limpias, desacopladas de los valores originales del dataset fuente.  
Todos los objetos incluyen `DROP IF EXISTS` para garantizar idempotencia (re-ejecutabilidad).

### **dim_categoria — Dimensión de Categorías de Producto**
Extrae los valores únicos de `product_category` y asigna un ID surrogate.

In [3]:
%%sql
DROP TABLE IF EXISTS dim_categoria;
CREATE TABLE dim_categoria AS
SELECT
    ROW_NUMBER() OVER (ORDER BY product_category) AS id_categoria,
    product_category
FROM (
    SELECT DISTINCT product_category
    FROM silver
) categorias;


StatementMeta(, 30bf5b80-3576-4ae1-9006-21f1139309a7, 6, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

## **Vista Previa — dim_categoria**
Verificación de que la dimensión fue creada correctamente con sus 
claves surrogadas (`id_categoria`) y valores únicos de `product_category`.

In [4]:
%%sql
select *
from dim_categoria

StatementMeta(, 30bf5b80-3576-4ae1-9006-21f1139309a7, 7, Finished, Available, Finished, False)

<Spark SQL result set with 3 rows and 2 fields>

### **dim_fecha — Dimensión Calendario**
   Extrae las fechas únicas de las transacciones y deriva atributos temporales: año, mes y día.

In [5]:
%%sql
DROP TABLE IF EXISTS dim_fecha;
CREATE  TABLE dim_fecha AS
SELECT
    ROW_NUMBER() OVER (ORDER BY date) AS id_fecha,
    date                              AS fecha,
    YEAR(date)                        AS anio,
    MONTH(date)                       AS mes,
    DAY(date)                         AS dia
FROM (
    SELECT DISTINCT date
    FROM silver
) fechas;

StatementMeta(, 30bf5b80-3576-4ae1-9006-21f1139309a7, 9, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

## **Vista Previa — dim_fecha**
Verificación de que la dimensión fue creada correctamente con su clave 
surrogada (`id_fecha`) y los atributos temporales derivados: 
`anio`, `mes` y `dia`.

In [6]:
%%sql
select *
from dim_fecha

StatementMeta(, 30bf5b80-3576-4ae1-9006-21f1139309a7, 10, Finished, Available, Finished, False)

<Spark SQL result set with 345 rows and 5 fields>

### **dim_genero — Dimensión de Géneros únicos de clientes**
Extrae los valores únicos de `gender` y asigna un ID surrogate.

In [7]:
%%sql

DROP TABLE IF EXISTS dim_genero;

CREATE TABLE dim_genero AS
SELECT
    ROW_NUMBER() OVER (
        ORDER BY gender
    )                AS id_genero,
    gender           AS nombre_genero
FROM (
    SELECT DISTINCT gender
    FROM silver
) generos;

StatementMeta(, 30bf5b80-3576-4ae1-9006-21f1139309a7, 12, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

## **Vista Previa — dim_genero**
Verificación de que la dimensión fue creada correctamente con su clave 
surrogada (`id_genero`) y los dos valores únicos de `nombre_genero` 
normalizados tras la limpieza de mayúsculas/minúsculas.

In [8]:
%%sql
select *
from dim_genero

StatementMeta(, 30bf5b80-3576-4ae1-9006-21f1139309a7, 13, Finished, Available, Finished, False)

<Spark SQL result set with 2 rows and 2 fields>

### **Hechos_ventas — Transacciones con métricas de venta**


In [9]:
%%sql
DROP TABLE IF EXISTS hechos_ventas;
CREATE TABLE hechos_ventas AS
SELECT
    dg.id_genero,
    s.transaction_id,
    s.customer_id,
    dc.id_categoria,
    df.id_fecha,
    s.quantity,
    s.price_per_unit,
    s.total_amount,
    s.age
FROM silver s
JOIN dim_genero dg
    ON s.gender = dg.nombre_genero
JOIN dim_categoria dc
    ON s.product_category = dc.product_category
JOIN dim_fecha df
    ON s.date = df.fecha;

StatementMeta(, 30bf5b80-3576-4ae1-9006-21f1139309a7, 15, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

## **Vista Previa — hechos_ventas**
Verificación de que la tabla de hechos fue construida correctamente.
Debe mostrar las claves foráneas (`id_genero`, `id_categoria`, `id_fecha`) 
resueltas desde las dimensiones y las métricas de negocio 
(`quantity`, `price_per_unit`, `total_amount`) sin valores nulos.

In [10]:
%%sql
SELECT *
from hechos_ventas

StatementMeta(, 30bf5b80-3576-4ae1-9006-21f1139309a7, 16, Finished, Available, Finished, True)

<Spark SQL result set with 1000 rows and 9 fields>